In [ ]:
import math
import re
from dataclasses import dataclass

import gradio as gr

def formatoCOP(valor: float) -> str:
    """Formatea un valor en COP con punto de miles y coma decimal."""
    texto = f"{valor:,.2f}"                       # 1,014,330.07
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    return f"$ {texto}"


# Registro de la tabla de proyección
@dataclass
class FilaProyeccion:
    mes: int
    intereses: float
    saldoFinal: float


# ---------------------------------------------------------------------------
# CAPA DE REGLAS: entidad CDT
# ---------------------------------------------------------------------------
class CDT:
    """Certificado de Depósito a Término: monto inicial y plazo en meses."""

    def __init__(self, pMontoInicial: float, pPlazo: int):
        self.montoInicial = pMontoInicial
        self.plazo = pPlazo

    # --- RF2: validaciones (devuelven el valor limpio o lanzan ValueError) ---
    @staticmethod
    def validarMonto(texto: str) -> float:
        try:
            monto = float(str(texto).strip().replace("$", "").replace(" ", ""))
        except ValueError:
            raise ValueError("El monto debe ser un número (no texto).")
        if math.isnan(monto) or math.isinf(monto):
            raise ValueError("El monto no es un número válido.")
        if monto <= 0:
            raise ValueError("El monto debe ser mayor a cero.")
        return monto

    @staticmethod
    def validarPlazo(texto: str) -> int:
        texto = str(texto).strip()
        try:
            valor = float(texto.replace(",", "."))
        except ValueError:
            raise ValueError("El plazo debe ser un número entero (no texto).")
        if valor <= 0:
            raise ValueError("El plazo debe ser mayor o igual a 1 mes.")
        if not re.fullmatch(r"\d+", texto):      # rechaza 12.5, 12,5, 12.0
            raise ValueError("El plazo debe ser un número entero (sin decimales).")
        return int(texto)

    # --- RF3: tasa mensual en % según el plazo ---
    def calcularTasaMensual(self) -> float:
        return 0.001695 * self.plazo + 0.0983

    # --- RF4: liquidación de un mes (sin redondear en los cálculos) ---
    def liquidarMes(self, pSaldoAnterior: float, pTasa: float) -> tuple:
        interes = pSaldoAnterior * (pTasa / 100)
        return interes, pSaldoAnterior + interes

    # --- RF5: tabla completa, comenzando con el mes 0 ---
    def proyectar(self) -> list:
        tasa = self.calcularTasaMensual()
        saldo = self.montoInicial
        filas = [FilaProyeccion(0, 0.0, saldo)]
        for mes in range(1, self.plazo + 1):
            interes, saldo = self.liquidarMes(saldo, tasa)
            filas.append(FilaProyeccion(mes, interes, saldo))
        return filas

    # --- RF6: resultado final ---
    def calcularResultado(self, pFilas: list) -> tuple:
        saldoFinal = pFilas[-1].saldoFinal
        totalIntereses = saldoFinal - self.montoInicial
        return saldoFinal, totalIntereses



class Presentacion:
    """Captura datos del usuario y muestra la proyección."""


    def capturarCDT(self, pMonto: str, pPlazo: str) -> CDT:
        monto = CDT.validarMonto(pMonto)
        plazo = CDT.validarPlazo(pPlazo)
        return CDT(monto, plazo)

    def presentarProyeccion(self, pCdt: CDT, pFilas: list) -> tuple:
        tasa = pCdt.calcularTasaMensual()
        saldoFinal, totalInt = pCdt.calcularResultado(pFilas)
        tabla = [[f.mes, formatoCOP(f.intereses), formatoCOP(f.saldoFinal)]
                 for f in pFilas]
        tasaTxt = f"{tasa:.4f}".replace(".", ",") + "%"
        resumen = (
            f"### ✅ Resultado\n"
            f"- **Tasa mensual aplicada:** {tasaTxt}\n"
            f"- **Total de intereses ganados:** {formatoCOP(totalInt)}\n"
            f"- **Al final del plazo la persona recibirá:** "
            f"**{formatoCOP(saldoFinal)}**"
        )
        return resumen, tabla

    def simular(self, pMonto: str, pPlazo: str) -> tuple:
        try:
            cdt = self.capturarCDT(pMonto, pPlazo)      # RF1 + RF2
        except ValueError as error:                     # dato inválido: se vuelve a solicitar
            return f"❌ **Error:** {error} Corrige el dato e inténtalo de nuevo.", []
        filas = cdt.proyectar()                         # RF3 + RF4 + RF5
        return self.presentarProyeccion(cdt, filas)     # RF5 + RF6

    def construirInterfaz(self) -> gr.Blocks:
        with gr.Blocks(theme=gr.themes.Soft(), title="Proyección CDT") as demo:
            gr.Markdown("# 🏦 Proyección de un CDT\n"
                        "Ingresa el monto inicial (COP) y el plazo en meses.")
            with gr.Row():
                with gr.Column(scale=1):
                    txtMonto = gr.Textbox(label="Monto inicial (COP)",
                                          placeholder="Ej: 1000000")
                    txtPlazo = gr.Textbox(label="Plazo (meses, entero ≥ 1)",
                                          placeholder="Ej: 12")
                    btn = gr.Button("Calcular proyección", variant="primary")
                    gr.Examples([["1000000", "12"], ["5000000", "24"],
                                 ["1000000", "1"], ["1000000", "60"]],
                                inputs=[txtMonto, txtPlazo])
                with gr.Column(scale=2):
                    salResumen = gr.Markdown()
                    salTabla = gr.Dataframe(
                        headers=["Mes", "Intereses", "Saldo final mes"],
                        datatype=["number", "str", "str"],
                        interactive=False)
            btn.click(self.simular, [txtMonto, txtPlazo], [salResumen, salTabla])
            txtPlazo.submit(self.simular, [txtMonto, txtPlazo], [salResumen, salTabla])
        return demo


def main():
    presentacion = Presentacion()
    presentacion.construirInterfaz().launch()


if __name__ == "__main__":
    main()

/tmp/ipykernel_1704/2012333304.py:118: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Proyección CDT") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e2ef2e9c7f6c7f481.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
